## Problem:
### For recommending the movies based on a single movie, Calculating similarity with 5 lakh+ movies and sorting them is computationally very high.

## Solution
### I used clustering, to assign the movies to different clusters where each cluster's movies will be simmilar. So, we don't need to calculate similarity between 5 lakh different movies for a recommendation, but we can just filtere ouut the movies of cluster and calculate similarity between them and sort in descending to recommend the best from the cluster.

Step 1: Import Packages and data

In [11]:
import pandas as pd
import numpy as np
from sklearn.cluster import MiniBatchKMeans
import time

In [4]:
df_vec=pd.read_csv("/content/text_vectors.csv")

Step 2: Load the Movie Vectors

In [5]:
def parse_vector(s):
    if not isinstance(s, str):
        return s
    s = s.strip('[]').replace('\n', ' ')
    try:
        return np.array([float(x) for x in s.split() if x], dtype=np.float32)
    except ValueError:
        return np.zeros(1, dtype=np.float32)

df_vec['rounded_vectors'] = df_vec['rounded_vectors'].apply(parse_vector)

Step 3: Prepare Data for Clustering

In [6]:
vector_matrix = np.vstack(df_vec['rounded_vectors'].values)

Step 4: Run K-Means Clustering

Since, we have more than 5 lakh movies, clustering them into 50000 clusters will in average place ~10 movies in each cluster.

In [10]:
num_clusters = 50000

In [8]:
kmeans = MiniBatchKMeans(
    n_clusters=num_clusters,
    batch_size=2048,
    random_state=42,
    n_init='auto'
)

cluster_labels = kmeans.fit_predict(vector_matrix)

Step 5: Save Results to CSV

In [9]:
df_recommendation = pd.DataFrame({
    'imdb_id': df_vec['imdb_id'],
    'cluster_label_new': cluster_labels
})

df_recommendation.to_csv("For_recommendation.csv", index=False)